## SPA: Model Serving

Este notebook comprueba el endpoint churn-classifier, que sirve la versión del modelo con el alias champion. El endpoint apunta a una versión concreta, así que si el alias pasa a otra versión hay que actualizarlo a mano. Antes de crearlo se comprobó que las dependencias guardadas con el modelo coinciden con las de requirements.txt y se revisó su firma. Para simular una consulta real se toman clientes de silver, se calculan sus features con silver_a_gold y se envían al endpoint. Las probabilidades que devuelve coinciden con las del modelo cargado en local. Un cliente al que le falta una columna recibe un error 400, porque la firma del modelo rechaza la entrada antes de que llegue a predecir.

Decisión: el endpoint recibe clientes en formato gold y quien lo llama calcula antes las features con silver_a_gold. Meter ese cálculo dentro del modelo haría el endpoint más robusto y queda como mejora.

Limitación: el endpoint necesita autenticación de Databricks, así que no se puede probar desde fuera sin una cuenta. (se hará, en todo caso, otro deploy para probar el modelo)

## ENG: Model Serving

This notebook tests the churn-classifier endpoint, which serves the model version holding the champion alias. The endpoint points to a specific version, so if the alias moves to another version the endpoint has to be updated manually. Before creating it, the dependencies stored with the model were checked against requirements.txt and its signature was reviewed. To simulate a real request, customers are taken from silver, their features are computed with silver_a_gold and they are sent to the endpoint. The probabilities it returns match those of the model loaded locally. A customer with a missing column receives a 400 error, because the model signature rejects the input before it reaches prediction.

Decision: the endpoint receives customers in gold format, and the caller computes the features beforehand with silver_a_gold. Moving that computation inside the model would make the endpoint more robust and is left as an improvement.

Limitation: the endpoint requires Databricks authentication, so it cannot be tested from outside without

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import json
import numpy as np
import mlflow
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

from churn.config import MODEL_NAME, SILVER_TABLE, ID, TARGET, ENDPOINT_NAME
from churn.features import silver_a_gold

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
uri_modelo = f"models:/{MODEL_NAME}/{champion.version}"

In [0]:
ruta = mlflow.pyfunc.get_model_dependencies(uri_modelo)
print(open(ruta).read())

In [0]:
info = mlflow.models.get_model_info(uri_modelo)
print(info.signature)

In [0]:
silver = spark.table(SILVER_TABLE).limit(5).toPandas()
X = silver_a_gold(silver).drop(columns=[ID, TARGET])

In [0]:
deploy = get_deploy_client("databricks")

registros = json.loads(X.to_json(orient="records"))
respuesta = deploy.predict(endpoint=ENDPOINT_NAME, inputs={"dataframe_records": registros})

respuesta

In [0]:
modelo = mlflow.pyfunc.load_model(uri_modelo)

local = modelo.predict(X)
remoto = np.array(respuesta["predictions"])

assert np.allclose(local, remoto), "El endpoint y el modelo local no coinciden"

In [0]:
X_mal = X.drop(columns=["contract"]).head(1)
registros_mal = json.loads(X_mal.to_json(orient="records"))

try:
    deploy.predict(endpoint=ENDPOINT_NAME, inputs={"dataframe_records": registros_mal})
except Exception as e:
    print(e)